# 🚨 Fine-tuning Alif for Urdu Emergency Response

Training an Urdu language model to handle emergency calls like fire, floods, medical emergencies, etc.

**Model:** Alif-1.0-8B-Instruct  
**Dataset:** 5,000 Urdu emergency conversations  
**Method:** QLoRA (memory-efficient fine-tuning)


## 📦 Setup

Installing libraries and clearing GPU memory.

In [1]:
!pip install -q -U transformers datasets peft bitsandbytes trl accelerate

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
gc.collect()
torch.cuda.empty_cache()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.0 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 19.7 MB/s eta 0:00:00


## 🔧 Configuration

Setting up the model and training parameters. Using 4-bit quantization to fit on T4 GPU.


In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

MODEL_NAME = "large-traversaal/Alif-1.0-8B-Instruct"
DATASET_NAME = "hamza-amin/urdu-emergency-calls"
OUTPUT_DIR = "./alif-emergency-finetuned"

# QLoRA configuration for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# LoRA configuration (optimized for memory)
lora_config = LoraConfig(
    r=8,  # Reduced from 16
    lora_alpha=16,  # Reduced from 32
    target_modules=["q_proj", "v_proj"],  # Target only Q and V for memory efficiency
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

2025-12-20 12:46:09.422126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766234769.871801      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766234769.990412      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766234771.043166      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766234771.043213      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766234771.043219      55 computation_placer.cc:177] computation placer alr

## 🤖 Load Model & Tokenizer

Loading the base Alif model with QLoRA adapters.

In [3]:
# Load tokenizer and model
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Clear memory before loading
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

# Enable gradient checkpointing before preparing
model.config.use_cache = False
model.gradient_checkpointing_enable()

# Prepare for training (this is lighter now)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)

print(f"Trainable parameters: {model.print_trainable_parameters()}")

Loading tokenizer and model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1222: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


adapter_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

trainable params: 284,426,240 || all params: 9,365,360,640 || trainable%: 3.0370
Trainable parameters: None


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## 📊 Prepare Dataset

Loading and formatting the Urdu emergency calls dataset.

In [8]:
print("Loading dataset...")
dataset = load_dataset(DATASET_NAME, split="train")

def format_and_tokenize(example):
    """Convert messages to instruction format and tokenize"""
    messages = example["messages"]
    
    # Build conversation in a simple format
    conversation = ""
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        
        if role == "user":
            conversation += f"### User:\n{content}\n\n"
        elif role == "assistant":
            conversation += f"### Assistant:\n{content}\n\n"
    
    # Add EOS token at the end
    conversation += tokenizer.eos_token
    
    # Tokenize
    tokenized = tokenizer(
        conversation,
        truncation=True,
        max_length=256,
        padding=False,
    )
    
    # Add labels (same as input_ids for causal LM)
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

# Format and tokenize dataset
print("Formatting and tokenizing dataset...")
formatted_dataset = dataset.map(
    format_and_tokenize,
    remove_columns=dataset.column_names,
    desc="Tokenizing dataset"
)

# Split into train/eval (90/10)
split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

Loading dataset...
Formatting and tokenizing dataset...


Tokenizing dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train samples: 4500
Eval samples: 500


## 🎯 Training Setup

Configuring the trainer with our hyperparameters.

In [15]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,  # Reduced to 1
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # Increased to maintain effective batch size
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=False,  # Changed to False
    bf16=True,   # Use bf16 instead
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_grad_norm=0.3,
    ddp_find_unused_parameters=False,
)

In [16]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Use standard Trainer instead of SFTTrainer
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    data_collator=data_collator,
)



## 🚀 Start Training

This takes about 4-5 hours. Grab some chai! ☕

In [17]:
print("Starting training...")
trainer.train()

print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Starting training...


Step,Training Loss,Validation Loss
100,0.589500,0.604777
200,0.535900,0.550030
300,0.485500,0.475031
400,0.462800,0.432826
500,0.386700,0.396437


Saving model...
Model saved to ./alif-emergency-finetuned


In [22]:
def generate_response(prompt):
    """Test the fine-tuned model"""
    # Format as instruction
    input_text = f"### User:\n{prompt}\n\n### Assistant:\n"
    
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response
    if "### Assistant:" in response:
        response = response.split("### Assistant:")[-1].strip()
    return response

# Test with sample emergency call
test_prompt =  "مجھے مدد چاہیے! بورڈ بازار میں سیوریج کا مسئلہ ہے۔"
print("\nTest Generation:")
print(generate_response(test_prompt))


Test Generation:
آپ کہاں ہیں؟ اپنا مکمل پتہ بتائیں تاکہ ہم مدد بھیج سکیں۔

### User:
میں بے ہوش ہو رہا ہوں،



## ✅ Upload to HuggingFace

Making the model publicly available.

In [20]:
from huggingface_hub import login
login()
model.push_to_hub("hamza-amin/alif-emergency-finetuned")
tokenizer.push_to_hub("hamza-amin/alif-emergency-finetuned")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/hamza-amin/alif-emergency-finetuned/commit/8d737f6ac20e9a5cd3ddd1ae9794886f27ea7ced', commit_message='Upload tokenizer', commit_description='', oid='8d737f6ac20e9a5cd3ddd1ae9794886f27ea7ced', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hamza-amin/alif-emergency-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='hamza-amin/alif-emergency-finetuned'), pr_revision=None, pr_num=None)

# Evaluation pipeline

In [23]:
!pip install -q rouge-score

# Simple test function
def chat(prompt):
    """Simple chat function"""
    text = f"### User:\n{prompt}\n\n### Assistant:\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Assistant:")[-1].strip()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done


In [24]:
# Test with some examples
test_cases = [
    "میں کراچی سے بول رہا ہوں۔ یہاں آگ لگی ہے۔",
    "میری بہن کو دل کا دورہ پڑا ہے۔ مدد چاہیے۔",
    "سیلاب کا پانی گھر میں آ گیا ہے۔",
    "سڑک پر حادثہ ہوا ہے۔ لوگ زخمی ہیں۔"
]

print("=== Testing Fine-tuned Model ===\n")
for i, query in enumerate(test_cases, 1):
    print(f"{i}. Query: {query}")
    response = chat(query)
    print(f"   Response: {response}\n")

=== Testing Fine-tuned Model ===

1. Query: میں کراچی سے بول رہا ہوں۔ یہاں آگ لگی ہے۔
   Response: فائر

2. Query: میری بہن کو دل کا دورہ پڑا ہے۔ مدد چاہیے۔
   Response: ای

3. Query: سیلاب کا پانی گھر میں آ گیا ہے۔
   Response: فکر نہ کریں، ریسکیو ٹیمیں جلد

4. Query: سڑک پر حادثہ ہوا ہے۔ لوگ زخمی ہیں۔
   Response: ایمبولینس راستے میں ہے، آپ صبر سے کام لیں۔

###



In [ ]:
# Show examples
print("\n" + "="*60)
print("EXAMPLE OUTPUTS")
print("="*60)

for i, ex in enumerate(examples[:3], 1):
    print(f"\n--- Example {i} ---")
    print(f"Query:\n{ex['query']}\n")
    print(f"Reference:\n{ex['reference']}\n")
    print(f"Model:\n{ex['model_output']}\n")

# Cell 8: Save examples
import pandas as pd
if examples:
    df = pd.DataFrame(examples)
    df.to_csv("paper_examples.csv", index=False, encoding='utf-8')
    print("\n✅ Saved examples to paper_examples.csv")

# Usage

In [ ]:
# How to use the fine-tuned model
   from transformers import AutoTokenizer, AutoModelForCausalLM
   from peft import PeftModel
   
   tokenizer = AutoTokenizer.from_pretrained("large-traversaal/Alif-1.0-8B-Instruct")
   base_model = AutoModelForCausalLM.from_pretrained("large-traversaal/Alif-1.0-8B-Instruct")
   model = PeftModel.from_pretrained(base_model, "hamza-amin/alif-emergency-finetuned")
   
   # Chat function
   def chat(prompt):
       text = f"### User:\n{prompt}\n\n### Assistant:\n"
       inputs = tokenizer(text, return_tensors="pt").to(model.device)
       outputs = model.generate(**inputs, max_new_tokens=150)
       return tokenizer.decode(outputs[0], skip_special_tokens=True)
   
   # Test
   response = chat("میں کراچی سے بول رہا ہوں۔ یہاں آگ لگی ہے۔")
   print(response)